## num_stages: Pipelining Loads and Compute

`num_stages` controls software pipelining inside a Triton program. In matmul-style kernels, the program repeatedly loads the next K tile and computes `tl.dot` on the current tile. Pipelining lets the compiler overlap those pieces of work.

More stages can hide memory latency, but they also use more registers and shared memory. Too many stages can reduce occupancy and make the kernel slower.

References: [triton.Config](https://triton-lang.org/main/python-api/generated/triton.Config.html), [tl.range](https://triton-lang.org/main/python-api/generated/triton.language.range.html)

## Mental Model

Imagine a matmul K-loop like this:

```python
for each K tile:
    load A tile and B tile
    dot(A tile, B tile)
```

With more pipeline stages, Triton can keep more future load work in flight while the current dot work is happening.

- Low `num_stages`: less buffering, lower resource use, but less latency hiding.
- High `num_stages`: more latency hiding, but higher register/shared-memory pressure.
- Elementwise kernels often do not benefit much because they do not have a deep repeated load/compute loop.

In [ ]:
import torch
import triton
import triton.language as tl
from triton.testing import do_bench

assert torch.cuda.is_available(), "This notebook needs a CUDA GPU."

## A Matmul Kernel With a K Loop

The K loop is the part where `num_stages` matters. We will keep the tile size and `num_warps` fixed, then sweep only `num_stages`.

In [ ]:
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    offsets_m = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offsets_n = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    offsets_k = tl.arange(0, BLOCK_SIZE_K)

    a_ptrs = a_ptr + offsets_m[:, None] * stride_am + offsets_k[None, :] * stride_ak
    b_ptrs = b_ptr + offsets_k[:, None] * stride_bk + offsets_n[None, :] * stride_bn
    c_ptrs = c_ptr + offsets_m[:, None] * stride_cm + offsets_n[None, :] * stride_cn

    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

    # Triton can software-pipeline this repeated load + dot loop.
    for k in range(0, K, BLOCK_SIZE_K):
        a_mask = (offsets_m[:, None] < M) & (offsets_k[None, :] + k < K)
        b_mask = (offsets_k[:, None] + k < K) & (offsets_n[None, :] < N)

        a = tl.load(a_ptrs + k * stride_ak, mask=a_mask, other=0.0)
        b = tl.load(b_ptrs + k * stride_bk, mask=b_mask, other=0.0)
        accumulator += tl.dot(a, b)

    c_mask = (offsets_m[:, None] < M) & (offsets_n[None, :] < N)
    tl.store(c_ptrs, accumulator, mask=c_mask)

## Launcher With num_stages

`num_stages` is passed at launch time, just like `num_warps`. It is not part of the kernel argument list.

In [ ]:
def matmul_with_stages(
    a: torch.Tensor,
    b: torch.Tensor,
    num_stages: int,
    block_m: int = 128,
    block_n: int = 128,
    block_k: int = 32,
) -> torch.Tensor:
    assert a.shape[1] == b.shape[0], "Incompatible dimensions"
    assert a.is_cuda and b.is_cuda

    M, K = a.shape
    K, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float32)

    grid = lambda META: (
        triton.cdiv(M, META["BLOCK_SIZE_M"]),
        triton.cdiv(N, META["BLOCK_SIZE_N"]),
    )

    matmul_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_SIZE_M=block_m,
        BLOCK_SIZE_N=block_n,
        BLOCK_SIZE_K=block_k,
        num_warps=4,
        num_stages=num_stages,
    )
    return c

## Benchmark Different Pipeline Depths

Each value compiles a specialized kernel. The benchmark does one correctness call before timing so compilation is not included in the reported time.

In [ ]:
def benchmark_num_stages(M: int = 1024, N: int = 1024, K: int = 1024):
    torch.manual_seed(0)
    a = torch.randn((M, K), device="cuda", dtype=torch.float16)
    b = torch.randn((K, N), device="cuda", dtype=torch.float16)

    expected = a @ b
    torch.cuda.synchronize()

    print(f"shape: A={tuple(a.shape)}, B={tuple(b.shape)}")
    print("fixed tile: BLOCK_M=128, BLOCK_N=128, BLOCK_K=32, num_warps=4")

    for num_stages in [2, 3, 4, 5]:
        c = matmul_with_stages(a, b, num_stages=num_stages)
        torch.cuda.synchronize()
        max_diff = torch.max(torch.abs(c - expected)).item()

        ms = do_bench(lambda: matmul_with_stages(a, b, num_stages=num_stages), warmup=25, rep=100)
        print(f"num_stages={num_stages:<2}  time={ms:.3f} ms  max_diff={max_diff:.3g}")


benchmark_num_stages()

## Reading the Results

If `num_stages=2` is slow, the program may not be hiding enough memory latency. If `num_stages=5` is slow, the extra buffering may be using too many resources and reducing occupancy.

The sweet spot often moves with `BLOCK_SIZE_K`: larger K tiles and heavier dot work can change how much pipelining helps.

## Using num_stages in Autotune

`num_stages` is usually tuned together with tile sizes and `num_warps`.

```python
@triton.autotune(
    configs=[
        triton.Config({"BLOCK_SIZE_M": 64,  "BLOCK_SIZE_N": 64,  "BLOCK_SIZE_K": 32}, num_warps=2, num_stages=3),
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 32}, num_warps=4, num_stages=3),
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 64}, num_warps=4, num_stages=4),
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 256, "BLOCK_SIZE_K": 32}, num_warps=8, num_stages=5),
    ],
    key=["M", "N", "K"],
)
@triton.jit
def matmul_kernel(...):
    ...
```

The important idea is that `num_stages` is not purely better when larger. It is a tradeoff between latency hiding and resource pressure.

## Exercise

Change `BLOCK_SIZE_K` from `32` to `64` in the launcher call and rerun the benchmark.

Then compare `K=1024` with `K=4096`. Deeper K loops usually make the pipeline behavior easier to observe.